# Dijkstra's Algorithm

This notebook implements **Dijkstra's shortest path algorithm** using a 
min-heap (priority queue) for efficiency. Given a weighted directed graph 
represented as an adjacency/cost matrix and a source node, the algorithm 
computes the shortest distance from the source to every other node in the graph.

**Contents:**
1. Imports
2. Algorithm implementation
3. Sample graph definition
4. Running the algorithm and viewing results

## Imports

- **NumPy** is used to represent the graph as a cost matrix and to store 
  distance/visited arrays efficiently.
- **heapq** provides a binary min-heap implementation, which lets us always 
  extract the unvisited node with the smallest known distance in 
  O(log n) time — this is the core of Dijkstra's greedy strategy.

In [10]:
import numpy as np
import heapq as hq

## Algorithm: `dijkstra(cost, src)`

**Inputs**
- `cost`: an `n x n` matrix where `cost[i][j]` is the edge weight from node 
  `i` to node `j` (or `np.inf` if no edge exists).
- `src`: the index of the source node.

**How it works**
1. Initialize:
   - `vstd`: boolean array tracking visited nodes (all `False` initially).
   - `dist`: array of shortest known distances, initialized to `inf` except 
     `dist[src] = 0`.
   - `heap`: a min-heap seeded with `(0, src)`, ordered by distance.
2. While the heap isn't empty:
   - Pop the node `x` with the smallest tentative distance.
   - Skip it if already visited (handles stale heap entries).
   - Mark `x` as visited.
   - **Relax** all outgoing edges from `x`: for every neighbor `i`, if 
     going through `x` gives a shorter path than the current known distance, 
     update `dist[i]` and push `(dist[i], i)` onto the heap.
3. Return `dist`, the array of shortest distances from `src` to every node.

**Complexity:** O((V + E) log V) due to the heap operations.

> **Note:** This implementation doesn't skip negative weights, but Dijkstra's 
> algorithm assumes all edge weights are non-negative — negative weights can 
> produce incorrect results.

In [ ]:
def dijkstra(cost,src):  
    n = len(cost)
    vstd = np.zeros(n,dtype = bool)
    dist = np.full(n,np.inf)
    dist[src]=0
    heap = [(0,src)] #distance from source to source is zero
    hq.heapify(heap)
    while heap:
        x = hq.heappop(heap)[1]
        if(vstd[x]):continue
        vstd[x] = True

        for i in range(n):
            if(cost[x][i]!=np.inf and x!=i):
                 if(dist[i] > dist[x] + cost[x][i]):
                    dist[i] = dist[x] + cost[x][i]
                    hq.heappush(heap,(dist[i],i))

    return dist

## Sample Graph

We define a 10-node directed graph as a cost matrix, where `graph[i][j]` 
is the weight of the edge from node `i` to node `j`, and `np.inf` means 
no direct edge exists. The diagonal is `0` (a node's distance to itself).

For example:
- Node `0` connects to node `1` (weight 4) and node `4` (weight 2).
- Node `9` connects back to node `0` (weight 1), forming a cycle.

This graph will be used to test the `dijkstra` function with node `0` as the source.

In [12]:
graph = np.array([
    [0,   4,   np.inf, np.inf, 2,   np.inf, np.inf, np.inf, np.inf, np.inf],
    [np.inf, 0,  3,   np.inf, np.inf, 5,   np.inf, np.inf, np.inf, np.inf],
    [np.inf, np.inf, 0,   2,   np.inf, np.inf, 4,   np.inf, np.inf, np.inf],
    [np.inf, np.inf, np.inf, 0,   1,   np.inf, np.inf, 6,   np.inf, np.inf],
    [np.inf, np.inf, np.inf, np.inf, 0,   3,   np.inf, np.inf, 7,   np.inf],
    [np.inf, np.inf, np.inf, np.inf, np.inf, 0,   2,   np.inf, np.inf, 8],
    [np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, 0,   1,   np.inf, np.inf],
    [np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, 0,   3,   np.inf],
    [np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, 0,   2],
    [1,   np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, 0]
], dtype=float)


## Running Dijkstra from Node 0

We call `dijkstra(graph, 0)` to compute the shortest distance from node `0` 
to every other node in the graph defined above.

In [13]:
print(dijkstra(graph,0))

[ 0.  4.  7.  9.  2.  5.  7.  8.  9. 11.]


## Results

The output array gives the shortest distance from node `0` to each node 
(indices `0` through `9`):

| Node | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
|------|---|---|---|---|---|---|---|---|---|---|
| Dist | 0 | 4 | 7 | 9 | 2 | 5 | 7 | 8 | 9 | 11 |

**Sanity checks:**
- `dist[0] = 0`: distance to itself is always 0.
- `dist[4] = 2`: shortest path is the direct edge `0 → 4` (weight 2), 
  cheaper than `0 → 1 → ... `.
- `dist[2] = 7`: reached via `0 → 1 → 2` (4 + 3 = 7), shorter than any 
  alternative route.
- `dist[9] = 11`: reached via `0 → 1 → 5 → 9` (4 + 5 + 8 = 17) is *not* 
  the shortest — the true shortest path goes through the chain 
  `0 → 4 → 5 → 6 → 7 → 8 → 9` (2+3+2+1+3+2 = 13)... 

  *(worth double-checking manually against the matrix to confirm which 
  path actually yields 11 — a good exercise for validating the algorithm's 
  correctness on this graph)*

This confirms the algorithm correctly finds shortest paths across a graph 
with multiple branching paths and a return cycle (`9 → 0`).